# TrustLens Day 2 — Deepfake Classifier (EfficientNet-B0) · Kaggle (clean build)

Transfer-learn EfficientNet-B0 on **GenImage**, hold out **one generator family** as out-of-distribution (OOD), and measure the **OOD accuracy drop**. Ends with a real-vs-AI inference demo.

### Do this first (right sidebar)
1. **Settings → Accelerator → GPU** (T4 x2 or P100).
2. **Settings → Internet → On** (needed for pip + pretrained weights).
3. **+ Add Input → Datasets tab**, attach 2+ GenImage generator subsets (paste URL):
   - `https://www.kaggle.com/datasets/vtphatt2/GenImage-BigGAN`  (held out as OOD)
   - `https://www.kaggle.com/datasets/vtphatt2/genimage-stable-diffusion-v1-4`  (training)
   - optional: `GenImage-wukong`, `GenImage-ADM`, … (more training families)

Then **Run All** top-to-bottom.

## 1. GPU check

In [ ]:
!nvidia-smi -L
import torch; print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())

## 2. Install TrustLens (forced clean install)
`--force-reinstall` is important: a plain `pip install` is a no-op if an older copy is already present, which leaves stale code. The version assert catches that.

In [ ]:
!pip -q install mlflow
!pip -q install --force-reinstall --no-deps git+https://github.com/umerjavaidkh/TrustLens.git
import trustlens
print("trustlens version:", trustlens.__version__)
assert trustlens.__version__ >= "0.2.0", (
    "Old package cached. Run -> Restart session, then re-run this cell." )
print("OK")

## 3. Index the attached GenImage datasets
Each generator family is detected from the folder above `train/val` — robust to Kaggle's wrapper folders. (Indexing ~600k+ files takes a minute or two.)

In [ ]:
from trustlens.models.splits import index_kaggle_inputs, list_generators
records = index_kaggle_inputs("/kaggle/input")
print(f"Indexed {len(records)} labelled images")
GENERATORS = list_generators(records)
print("GENERATORS =", GENERATORS)
assert len(GENERATORS) >= 2, "Attach >=2 GenImage generator datasets via + Add Input."

def pick_ood(gens, prefer="biggan"):
    for g in gens:
        if prefer in g.lower():
            return g
    return gens[0]

OOD_GEN = pick_ood(GENERATORS)
print("OOD_GENERATOR =", OOD_GEN, " (edit to hold out a different family)")

## 4. Config

In [ ]:
from trustlens.training.train import TrainConfig
cfg = TrainConfig(
    data_root="/kaggle/input",
    kaggle_inputs=True,
    ood_generator=OOD_GEN,
    img_size=224,
    batch_size=64,
    head_epochs=3,                # Phase 1: frozen backbone
    finetune_epochs=5,            # Phase 2: unfreeze last blocks
    unfreeze_blocks=2,
    max_per_class_per_gen=4000,   # raise (or None) for better numbers if you have GPU time
    target_fpr=0.01,
    experiment="trustlens-deepfake",
    out_dir="/kaggle/working/models",
)
cfg

## 5. Train (two-phase) + evaluate ID vs OOD

In [ ]:
import mlflow
mlflow.set_tracking_uri("file:/kaggle/working/mlruns")
from trustlens.training.train import train
model, report, logs = train(cfg)

## 6. The distribution-shift exhibit

In [ ]:
import json, matplotlib.pyplot as plt
print(report.summary_line())
print(json.dumps(report.as_dict(), indent=2))

labels = ["ID test", f"OOD\n({report.ood_generator})"]
accs = [report.id_test.accuracy, report.ood.accuracy]
plt.bar(labels, accs, color=["#2a9d8f", "#e76f51"]); plt.ylim(0, 1)
plt.ylabel("accuracy")
plt.title(f"OOD accuracy drop = {report.accuracy_drop:.3f} ({report.relative_drop:.0%})")
for i, a in enumerate(accs): plt.text(i, a + 0.01, f"{a:.3f}", ha="center")
plt.show()

## 7. MLflow metrics

In [ ]:
run = mlflow.search_runs(experiment_names=[cfg.experiment]).iloc[0]
print(run[[c for c in run.index if c.startswith("metrics.")]])

## 8. Inference demo — real vs AI
Load the trained model and score one real and one fake image: output is `P(fake)`, thresholded to a label.

In [ ]:
import torch
from PIL import Image
from trustlens.models.transforms import eval_transforms

device = "cuda" if torch.cuda.is_available() else "cpu"
model.eval()
tf = eval_transforms(cfg.img_size)
THRESHOLD = 0.5   # tune via metrics.evaluate_at_fpr for a target FPR

def predict(path):
    img = Image.open(path).convert("RGB")
    x = tf(img).unsqueeze(0).to(device)
    with torch.no_grad():
        return float(torch.softmax(model(x), 1)[0, 1])

real_path = next(r.path for r in records if r.label == 0)
fake_path = next(r.path for r in records if r.label == 1)
for name, pth in [("real sample", real_path), ("fake sample", fake_path)]:
    p = predict(pth)
    verdict = "FAKE / AI" if p >= THRESHOLD else "REAL"
    print(f"{name}: P(fake)={p:.3f} -> {verdict}")

## 9. Save the model

In [ ]:
import os
print("Best checkpoint:", "/kaggle/working/models/efficientnet_b0_best.pt")
print(os.listdir("/kaggle/working/models"))
# Everything in /kaggle/working is downloadable from the Output tab, or Save Version.

## Notes

- **real vs AI-generated**, specifically (that's what GenImage labels) — not face-swap / photoshop detection.
- **OOD drop is the headline:** high accuracy on the seen generator, lower on the held-out one — how much the detector degrades on an unseen generator.
- **Better numbers:** attach more generator datasets and/or raise `max_per_class_per_gen` and `finetune_epochs`.
- **Different hold-out:** set `OOD_GEN` (cell 3) to any name in `GENERATORS`.